# Smoke hai nguồn vector vùng
Chọn 1 GPU T4, bật Internet và secret **codex** có quyền đọc v136. Hoặc điền đường dẫn NPZ chính xác. Gửi **regionclip_slot_smoke136/result.json**.


In [ ]:
EXPECTED_SHA='b2f185999077670a27c50305d6e128a182cd451b25bdb625fe7e21a2e9995533'

from pathlib import Path
import hashlib,json,os,subprocess,sys,tempfile
import numpy as np

VERSION='ducanh3/notebook7bc83c4c76/136'
SECRET_NAME='codex'
MOUNTED_NPZ=''  # Optional exact paired_slots.npz mounted as input.
if MOUNTED_NPZ:
    source=Path(MOUNTED_NPZ)
else:
    from kaggle_secrets import UserSecretsClient
    root=Path(tempfile.mkdtemp(prefix='regionclip-smoke-'))
    cli=root/'cli'
    subprocess.run([sys.executable,'-m','pip','install','-q','--target',str(cli),'kaggle==2.2.5'],
                   check=True,timeout=300)
    env=dict(os.environ)
    for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):
        env.pop(key,None)
    env['KAGGLE_API_TOKEN']=UserSecretsClient().get_secret(SECRET_NAME).strip()
    env['PYTHONPATH']=str(cli)
    output=root/'output'
    output.mkdir()
    subprocess.run([sys.executable,'-c','from kaggle.cli import main; main()',
                    'kernels','output',VERSION,'-p',str(output),
                    '--file-pattern',r'(^|/)regionclip_slot_pair160/paired_slots[.]npz$'],
                   env=env,check=True,timeout=300)
    source,=output.rglob('paired_slots.npz')
with source.open('rb') as stream:
    digest=hashlib.file_digest(stream,'sha256').hexdigest()
if digest!=EXPECTED_SHA:
    raise ValueError('Paired cache SHA differs from audited v136')
with np.load(source,allow_pickle=False) as cache:
    data={name:cache[name] for name in ('coco_ids','clip','regionclip','boxes','valid')}
expected={'coco_ids':(160,), 'clip':(160,8,768),'regionclip':(160,8,1024),'boxes':(160,8,4),'valid':(160,8)}
if any(data[k].shape!=shape for k,shape in expected.items()):
    raise ValueError('Unexpected cache shapes')
if len(set(map(int,data['coco_ids'])))!=160 or data['valid'].dtype!=np.bool_ or data['valid'].sum()!=523:
    raise ValueError('Unexpected cache coverage or mask')
for name in ('clip','regionclip','boxes'):
    if data[name].dtype!=np.float32 or not np.isfinite(data[name]).all() or np.any(data[name][~data['valid']]):
        raise ValueError(f'Invalid values or padding: {name}')


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab: int
    dim: int = 512
    heads: int = 8
    hidden: int = 2048
    prototypes: int = 150
    layers: int = 3
    dropout: float = .2
    raw_dim: int = 768

def zero_pad(x, valid):
    return x.masked_fill(~valid[..., None], 0)

class FFN(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(c.dim, c.hidden), nn.ReLU(), nn.Dropout(c.dropout),
                                 nn.Linear(c.hidden, c.dim), nn.Dropout(c.dropout))
        self.norm = nn.LayerNorm(c.dim, eps=1e-5)

    def forward(self, x):
        return self.norm(x + self.net(x))

class DecoderLayer(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)
        self.object_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)
        self.relation_attn = nn.MultiheadAttention(c.dim, c.heads, c.dropout, batch_first=True)
        self.norm = nn.LayerNorm(c.dim)
        self.drop = nn.Dropout(c.dropout)
        self.gate_o, self.gate_r = nn.Linear(2 * c.dim, c.dim), nn.Linear(2 * c.dim, c.dim)
        self.ffn = FFN(c)

    def forward(self, x, previous_word, vo, vr, text_valid, region_valid):
        causal = torch.ones(x.shape[1], x.shape[1], device=x.device, dtype=torch.bool).triu(1)
        h = self.self_attn(x, x, x, attn_mask=causal, key_padding_mask=~text_valid, need_weights=False)[0]
        h = zero_pad(self.norm(x + self.drop(h)), text_valid)
        co = self.object_attn(h, vo, vo, key_padding_mask=~region_valid, need_weights=False)[0]
        cr = self.relation_attn(h, vr, vr, key_padding_mask=~region_valid, need_weights=False)[0]
        go = self.gate_o(torch.cat((co, previous_word), dim=-1)).sigmoid()
        gr = self.gate_r(torch.cat((cr, previous_word), dim=-1)).sigmoid()
        return zero_pad(self.ffn(go * co + gr * cr), text_valid)

class PromptMemory(nn.Module):
    def __init__(self,input_dim):
        super().__init__();self.slot=nn.Sequential(nn.Linear(input_dim,512),nn.ReLU(),nn.Linear(512,512))
        self.attention=nn.MultiheadAttention(512,8,dropout=0,batch_first=True)
        nn.init.zeros_(self.attention.out_proj.weight);nn.init.zeros_(self.attention.out_proj.bias)
    def forward(self,visual,slot_inputs,support,valid):
        slots=self.slot(slot_inputs)
        allowed=support.transpose(1,2).gt(0)&valid[:,None,:]
        active=allowed.any(-1)
        # Give inactive queries one finite key, then zero their entire residual.
        safe=allowed.clone();safe[:,:,0]|=~active
        mask=(~safe)[:,None].expand(-1,8,-1,-1).reshape(-1,visual.shape[1],slots.shape[1])
        delta=self.attention(visual,slots,slots,attn_mask=mask,need_weights=False)[0]
        return visual+delta.masked_fill(~active[...,None],0)

def box_weights(value):
    b=np.asarray(value,np.float64)
    if b.shape!=(4,) or not np.isfinite(b).all() or (b<0).any() or (b>1).any() or (b[2:]<=b[:2]).any():raise ValueError('Invalid normalized xyxy box')
    x0,y0,x1,y1=b*14;axis=np.arange(14,dtype=np.float64)
    wx=np.maximum(0,np.minimum(axis+1,x1)-np.maximum(axis,x0));wy=np.maximum(0,np.minimum(axis+1,y1)-np.maximum(axis,y0));weights=(wy[:,None]*wx[None,:]).reshape(196)
    if weights.sum()<=0:raise ValueError('Box has no patch support')
    return (weights/weights.sum()).astype(np.float32)

if not torch.cuda.is_available():
    raise RuntimeError('Select a Kaggle GPU accelerator')
device=torch.device('cuda:0')
torch.manual_seed(136)
torch.backends.cuda.matmul.allow_tf32=False
indices=np.flatnonzero((data['valid'].sum(1)>0)&(data['valid'].sum(1)<8))[:4]
if len(indices)!=4:
    raise ValueError('Need four rows with both active and padded slots')
valid=torch.from_numpy(data['valid'][indices]).to(device)
boxes=torch.from_numpy(data['boxes'][indices]).to(device)
weights=np.zeros((4,8,196),np.float32)
for i,j in zip(*np.nonzero(data['valid'][indices])):
    weights[i,j]=box_weights(data['boxes'][indices[i],j])
support=torch.from_numpy(weights).to(device)
visual=torch.randn(4,196,512,device=device)
relation=torch.randn_like(visual)
text=torch.randn(4,8,512,device=device)
prefix=torch.randn(4,5,512,device=device)
tokens=torch.randint(0,64,(4,5),device=device)
prefix_mask=torch.ones(4,5,dtype=torch.bool,device=device)
patch_mask=torch.ones(4,196,dtype=torch.bool,device=device)
decoder=DecoderLayer(Config(vocab=64,dropout=0)).to(device).eval()
head=nn.Linear(512,64).to(device).eval()
decoder.requires_grad_(False)
head.requires_grad_(False)
template=PromptMemory(1028).to(device)
from copy import deepcopy

def forward(values, projection, adapter):
    slots=torch.cat((text,projection(values),boxes),dim=-1)
    updated=adapter(visual,slots,support,valid)
    return head(decoder(prefix,prefix,updated,relation,prefix_mask,patch_mask)),updated

def grad_norm(param):
    if param.grad is None or not torch.isfinite(param.grad).all():
        raise ValueError('Missing or nonfinite gradient')
    return float(param.grad.norm().item())

results={}
initial={}
for source_name in ('clip','regionclip'):
    values=torch.from_numpy(data[source_name][indices]).to(device)
    projection=nn.Linear(values.shape[-1],512,bias=False).to(device)
    adapter=deepcopy(template)
    logits,updated=forward(values,projection,adapter)
    torch.testing.assert_close(updated,visual,rtol=0,atol=0)
    initial[source_name]=logits.detach()
    loss=F.cross_entropy(logits.flatten(0,1),tokens.flatten())
    loss.backward()
    initial_adapter_grad=grad_norm(adapter.attention.out_proj.weight)
    if initial_adapter_grad<=0:
        raise ValueError('Zero first-step residual gradient')
    with torch.no_grad():
        adapter.attention.out_proj.weight.add_(adapter.attention.out_proj.weight.grad,alpha=-.01)
    adapter.zero_grad(set_to_none=True)
    projection.zero_grad(set_to_none=True)
    logits,updated=forward(values,projection,adapter)
    with torch.no_grad():
        changed=values.clone()
        changed[~valid]=42
        padding_logits,_=forward(changed,projection,adapter)
        torch.testing.assert_close(logits,padding_logits,rtol=0,atol=0)
    second_loss=F.cross_entropy(logits.flatten(0,1),tokens.flatten())
    second_loss.backward()
    projection_grad=grad_norm(projection.weight)
    if projection_grad<=0:
        raise ValueError('Zero gradient in feature projection')
    results[source_name]=dict(parameters=projection.weight.numel(),
                              initial_xe=float(loss.item()),residual_gradient=initial_adapter_grad,
                              projection_gradient=projection_grad,padded_values_invariant=True)
torch.testing.assert_close(initial['clip'],initial['regionclip'],rtol=0,atol=0)
out=Path('/kaggle/working/regionclip_slot_smoke136')
out.mkdir(exist_ok=True)
result=dict(source=VERSION,paired_sha256=digest,device=torch.cuda.get_device_name(0),
            image_ids=data['coco_ids'][indices].tolist(),branches=results,initial_logits_equal=True,
            synthetic_context_and_targets=True)
(out/'result.json').write_text(json.dumps(result,indent=2,allow_nan=False),encoding='utf8')
print(json.dumps(result,indent=2))
print('SEND:',out/'result.json')
